In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

from tensorflow.keras.applications import MobileNetV2

In [2]:
IMG_SIZE = 64
MAX_FRAMES = 20

DATASET_PATH = r"C:\Users\Anagha Kesav\Desktop\Deep Fake Vdo Prediction\dataset"

REAL_PATH = os.path.join(DATASET_PATH, "videos_real")
FAKE_PATH = os.path.join(DATASET_PATH, "videos_fake")

In [3]:
def extract_frames(video_path):

    frames = []

    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    skip = max(total_frames // MAX_FRAMES, 1)

    count = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if count % skip == 0:

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))

            frame = frame.astype("float32") / 255.0

            frames.append(frame)

        count += 1

        if len(frames) >= MAX_FRAMES:
            break

    cap.release()

    return frames

In [4]:
X = []
y = []

print("Loading REAL videos...")

for file in tqdm(os.listdir(REAL_PATH)):

    video_path = os.path.join(REAL_PATH, file)

    try:

        frames = extract_frames(video_path)

        for frame in frames:

            X.append(frame)

            y.append(0)   # REAL

    except Exception as e:

        print("ERROR:", e)

print("Loading FAKE videos...")

for file in tqdm(os.listdir(FAKE_PATH)):

    video_path = os.path.join(FAKE_PATH, file)

    try:

        frames = extract_frames(video_path)

        for frame in frames:

            X.append(frame)

            y.append(1)   # FAKE

    except Exception as e:

        print("ERROR:", e)

Loading REAL videos...


100%|██████████| 53/53 [00:14<00:00,  3.57it/s]


Loading FAKE videos...


100%|██████████| 53/53 [00:12<00:00,  4.34it/s]


In [5]:
X = np.array(X)
y = np.array(y)

print("Dataset Loaded")

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("Real Frames:", np.sum(y == 0))
print("Fake Frames:", np.sum(y == 1))

Dataset Loaded
X Shape: (2120, 64, 64, 3)
y Shape: (2120,)
Real Frames: 1060
Fake Frames: 1060


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
y_train = to_categorical(y_train, 2)
y_test = to_categorical(y_test, 2)

In [8]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(64,64,3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(2, activation='softmax')

])

C:\Users\Anagha Kesav\AppData\Local\Temp\ipykernel_12984\3900191369.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,430,338 (9.27 MB)

 Trainable params: 172,354 (673.26 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

Epoch 1/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.4829 - loss: 0.8676 - val_accuracy: 0.5212 - val_loss: 0.7057
Epoch 2/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.5106 - loss: 0.7348 - val_accuracy: 0.5118 - val_loss: 0.6949
Epoch 3/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.5436 - loss: 0.7098 - val_accuracy: 0.5354 - val_loss: 0.6862
Epoch 4/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.5637 - loss: 0.6831 - val_accuracy: 0.5472 - val_loss: 0.6738
Epoch 5/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.5719 - loss: 0.6694 - val_accuracy: 0.5401 - val_loss: 0.6625
Epoch 6/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.5920 - loss: 0.6529 - val_accuracy: 0.5731 - val_loss: 0.6485
Epoch 7/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.6262 - loss: 0.6431 - val_accuracy: 0.5802 - val_loss: 0.6397
Epoch 8/20
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6238 - loss: 0.6091 - val_acc

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nAccuracy:", accuracy * 100)

# =====================================================
# SAVE MODEL
# =====================================================

model.save("deepfake_model.h5")

print("\nModel Saved Successfully")

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.6179 - loss: 0.6362



Accuracy: 61.79245114326477

Model Saved Successfully
